In [1]:
!pip install -q openai chromadb sentence-transformers

In [2]:
import openai
import chromadb
from sentence_transformers import SentenceTransformer

print("All libraries installed successfully! 🚀")

All libraries installed successfully! 🚀


Load API Key and RAG Environment

In [6]:
# Install dependencies
!pip install -q chromadb==1.0.20 sentence-transformers openai "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0"

# Imports
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

# Initialize Groq client (NOT OpenAI)
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load local ChromaDB
db_client = chromadb.PersistentClient(path="/content/chroma_db")
collection = db_client.get_collection("negative_reviews")

print("Recovery completed successfully! 🚀")
print("Documents in collection:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TypeError: RustBindingsAPI.create_collection: `schema` is not present.

Rebuild ChromaDB Collection

In [ ]:
import shutil
import os

# Path to corrupted ChromaDB folder
chroma_path = os.path.join(DATA_PATH, "chroma_db")

# Delete corrupted database
if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)
    print("Old corrupted ChromaDB deleted successfully! 🗑️")

# Recreate ChromaDB client and collection
db_client = chromadb.PersistentClient(path=chroma_path)

collection = db_client.get_or_create_collection(
    name="negative_reviews"
)

print("Fresh ChromaDB collection created successfully! 🚀")

Re-Add Documents to ChromaDB

In [ ]:
import pandas as pd

In [ ]:
# Prepare documents (same as before)
negative_path = os.path.join(DATA_PATH, "negative_reviews.csv")
negative_df = pd.read_csv(negative_path)

sample_size = 5000

documents = (
    negative_df["reviewText"]
    .dropna()
    .astype(str)
    .head(sample_size)
    .tolist()
)

# Generate embeddings again if not already available
# (Skip this if `embeddings` variable still exists)
if "embeddings" not in globals():
    embeddings = model.encode(documents, show_progress_bar=True)

# Create unique IDs
ids = [f"doc_{i}" for i in range(len(documents))]

# Add to ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids
)

print("Documents re-added successfully! 🚀")
print("Total documents in collection:", collection.count())

In [ ]:
import pandas as pd
import os
import shutil
import chromadb

# ---------------------------------------
# 1. Use local writable directory
# ---------------------------------------
LOCAL_CHROMA_PATH = "/content/chroma_db"

# Remove old local database if it exists
if os.path.exists(LOCAL_CHROMA_PATH):
    shutil.rmtree(LOCAL_CHROMA_PATH)
    print("Old local ChromaDB removed.")

# ---------------------------------------
# 2. Create fresh local ChromaDB
# ---------------------------------------
db_client = chromadb.PersistentClient(path=LOCAL_CHROMA_PATH)

collection = db_client.get_or_create_collection(
    name="negative_reviews"
)

print("Fresh local ChromaDB created successfully! 🚀")

# ---------------------------------------
# 3. Load negative reviews
# ---------------------------------------
negative_path = os.path.join(DATA_PATH, "negative_reviews.csv")
negative_df = pd.read_csv(negative_path)

sample_size = 5000

documents = (
    negative_df["reviewText"]
    .dropna()
    .astype(str)
    .head(sample_size)
    .tolist()
)

print("Documents loaded:", len(documents))

# ---------------------------------------
# 4. Generate embeddings if needed
# ---------------------------------------
if "embeddings" not in globals():
    embeddings = model.encode(documents, show_progress_bar=True)

# ---------------------------------------
# 5. Create IDs
# ---------------------------------------
ids = [f"doc_{i}" for i in range(len(documents))]

# ---------------------------------------
# 6. Add to ChromaDB
# ---------------------------------------
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids
)

print("Documents added successfully! 🎉")
print("Total documents in collection:", collection.count())
print("Database location:", LOCAL_CHROMA_PATH)

In [ ]:
LOCAL_CHROMA_PATH = "/content/chroma_db"

Retrieve Relevant Reviews

In [ ]:
def retrieve_reviews(query, n_results=5):
    """
    Retrieve the most relevant customer reviews
    from ChromaDB using semantic search.
    """

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Search vector database
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    # Return retrieved documents
    return results["documents"][0]

Build the AI Business Analyst

In [ ]:
def ask_business_analyst(question, n_results=5):
    """
    Answer business questions using:
    1. Semantic search (ChromaDB)
    2. OpenAI LLM
    """

    # Retrieve relevant reviews
    retrieved_docs = retrieve_reviews(question, n_results=n_results)

    # Build context
    context = "\n\n".join([
        f"Review {i+1}:\n{doc}"
        for i, doc in enumerate(retrieved_docs)
    ])

    # Create prompt
    prompt = f"""
You are a senior business analyst.

Analyze the following customer reviews and answer the question.

Question:
{question}

Customer Reviews:
{context}

Provide your response in the following format:

1. Executive Summary
2. Key Complaint Themes
3. Business Recommendations
4. Priority Actions
"""

    # Call OpenAI API
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    answer = response.output_text

    print(answer)
    return answer

Ask Your First Business Question

In [ ]:
ask_business_analyst(
    "What are the top customer complaints and what should the company improve first?"
)

In [ ]:
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")
print("Groq API key loaded successfully! 🚀")

Initialize Groq Client

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Load Groq API key from Colab Secrets
groq_api_key = userdata.get("GROQ_API_KEY")

# Initialize Groq client (OpenAI-compatible)
client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client initialized successfully! 🚀")

Replace ask_business_analyst() function

In [ ]:
def ask_business_analyst(question, n_results=5):
    """
    Answer business questions using:
    1. Semantic search (ChromaDB)
    2. Groq LLM
    """

    # Retrieve relevant reviews
    retrieved_docs = retrieve_reviews(question, n_results=n_results)

    # Build context
    context = "\n\n".join([
        f"Review {i+1}:\n{doc}"
        for i, doc in enumerate(retrieved_docs)
    ])

    # Create prompt
    prompt = f"""
You are a senior business analyst.

Analyze the following customer reviews and answer the question.

Question:
{question}

Customer Reviews:
{context}

Provide your response in the following format:

1. Executive Summary
2. Key Complaint Themes
3. Business Recommendations
4. Priority Actions
"""

    # Call Groq API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    print(answer)
    return answer

Retrieve Function

In [ ]:
def retrieve_reviews(query, n_results=5):
    """
    Retrieve the most relevant customer reviews
    from ChromaDB using semantic search.
    """

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Search vector database
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )

    # Return top matching documents
    return results["documents"][0]

Load the Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully! 🚀")

Recreate collection

In [ ]:
!pip install -q chromadb sentence-transformers openai

In [ ]:
import chromadb

# Path of local ChromaDB created yesterday
LOCAL_CHROMA_PATH = "/content/chroma_db"

# Connect to existing local ChromaDB
db_client = chromadb.PersistentClient(path=LOCAL_CHROMA_PATH)

# Load collection
collection = db_client.get_collection("negative_reviews")

print("Collection loaded successfully! 🚀")
print("Documents in collection:", collection.count())

Test your RAG System

In [ ]:
ask_business_analyst(
    "What are the top customer complaints and what should the company improve first?"
)

In [7]:
# ==========================================
# COMPLETE WORKING RAG SYSTEM (WITHOUT CHROMADB)
# ==========================================

!pip install -q sentence-transformers openai scikit-learn

import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from google.colab import userdata

# ------------------------------------------
# 1. Initialize Groq Client
# ------------------------------------------
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# ------------------------------------------
# 2. Load Embedding Model
# ------------------------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# ------------------------------------------
# 3. Load Negative Reviews
# ------------------------------------------
PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"
DATA_PATH = os.path.join(PROJECT_PATH, "data")

negative_path = os.path.join(DATA_PATH, "negative_reviews.csv")
negative_df = pd.read_csv(negative_path)

# Use first 5000 reviews
documents = (
    negative_df["reviewText"]
    .dropna()
    .astype(str)
    .head(5000)
    .tolist()
)

print("Documents loaded:", len(documents))

# ------------------------------------------
# 4. Generate Document Embeddings
# ------------------------------------------
print("Generating embeddings...")
document_embeddings = model.encode(documents, show_progress_bar=True)

print("Embeddings generated successfully! 🚀")

# ------------------------------------------
# 5. Semantic Retrieval Function
# ------------------------------------------
def retrieve_reviews(query, n_results=5):
    # Embed query
    query_embedding = model.encode([query])

    # Compute cosine similarity
    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Get top indices
    top_indices = similarities.argsort()[-n_results:][::-1]

    # Return top documents
    return [documents[i] for i in top_indices]

# ------------------------------------------
# 6. RAG Question Answering Function
# ------------------------------------------
def ask_business_analyst(question, n_results=5):
    # Retrieve relevant reviews
    retrieved_docs = retrieve_reviews(question, n_results)

    # Build context
    context = "\n\n".join(
        [f"Review {i+1}:\n{doc}" for i, doc in enumerate(retrieved_docs)]
    )

    # Prompt
    prompt = f"""
You are a senior business analyst.

Analyze the following customer reviews and answer the question.

Question:
{question}

Customer Reviews:
{context}

Provide:
1. Executive Summary
2. Key Complaint Themes
3. Business Recommendations
4. Priority Actions
"""

    # Generate response using Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    print(answer)
    return answer

print("RAG system initialized successfully! 🤖🚀")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents loaded: 5000
Generating embeddings...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings generated successfully! 🚀
RAG system initialized successfully! 🤖🚀


Test the System

In [8]:
ask_business_analyst(
    "What are the top customer complaints and what should the company improve first?"
)

**1. Executive Summary**

The customer reviews analyzed reveal significant dissatisfaction with the game Arcanum, primarily due to technical issues, bugs, and poor customer support. The reviews highlight the game's potential but express frustration with the numerous problems that hinder the gaming experience. The company, Troika, is criticized for releasing a flawed product, and customers are seeking improvements and better support. In contrast, some reviews mention other games, such as Everquest and Asheron's Call, which are used as benchmarks for comparison.

**2. Key Complaint Themes**

1. **Technical Issues**: Bugs, glitches, and crashes that prevent players from progressing through the game.
2. **Poor Customer Support**: Lack of effective communication, unhelpful patches, and unresponsive developers.
3. **Gameplay Limitations**: Forced group play, inability to hunt solo, and limited options for players who prefer solo gameplay.
4. **Value for Money**: Some customers feel that the 

"**1. Executive Summary**\n\nThe customer reviews analyzed reveal significant dissatisfaction with the game Arcanum, primarily due to technical issues, bugs, and poor customer support. The reviews highlight the game's potential but express frustration with the numerous problems that hinder the gaming experience. The company, Troika, is criticized for releasing a flawed product, and customers are seeking improvements and better support. In contrast, some reviews mention other games, such as Everquest and Asheron's Call, which are used as benchmarks for comparison.\n\n**2. Key Complaint Themes**\n\n1. **Technical Issues**: Bugs, glitches, and crashes that prevent players from progressing through the game.\n2. **Poor Customer Support**: Lack of effective communication, unhelpful patches, and unresponsive developers.\n3. **Gameplay Limitations**: Forced group play, inability to hunt solo, and limited options for players who prefer solo gameplay.\n4. **Value for Money**: Some customers feel

In [9]:
# ==========================================================
# Create app/streamlit_app.py with complete Streamlit code
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# Project paths
PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"
APP_PATH = os.path.join(PROJECT_PATH, "app")

# Create app folder if it doesn't exist
os.makedirs(APP_PATH, exist_ok=True)

# Full file path
file_path = os.path.join(APP_PATH, "streamlit_app.py")

# Complete Streamlit application code
streamlit_code = '''
import os
import pandas as pd
import streamlit as st
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# -----------------------------
# Page Configuration
# -----------------------------
st.set_page_config(
    page_title="Customer Review Insight Engine",
    page_icon="📊",
    layout="wide"
)

# -----------------------------
# Title and Description
# -----------------------------
st.title("📊 Customer Review Insight Engine")
st.markdown(
    "AI-powered business insights from customer reviews using "
    "Semantic Search + RAG + Groq LLM."
)

# -----------------------------
# Load API Key from Streamlit Secrets
# -----------------------------
groq_api_key = st.secrets["GROQ_API_KEY"]

# Initialize Groq client (OpenAI-compatible)
client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# -----------------------------
# Load Data and Models
# -----------------------------
@st.cache_resource
def load_resources():
    # Project root = parent of app/
    project_path = os.path.dirname(os.path.dirname(__file__))
    data_path = os.path.join(project_path, "data", "negative_reviews.csv")

    # Load dataset
    df = pd.read_csv(data_path)

    # Use first 5000 negative reviews
    documents = (
        df["reviewText"]
        .dropna()
        .astype(str)
        .head(5000)
        .tolist()
    )

    # Load embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate embeddings
    embeddings = model.encode(
        documents,
        show_progress_bar=False
    )

    return documents, model, embeddings

documents, model, document_embeddings = load_resources()

# -----------------------------
# Semantic Search Function
# -----------------------------
def retrieve_reviews(query, n_results=5):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    top_indices = similarities.argsort()[-n_results:][::-1]

    return [documents[i] for i in top_indices]

# -----------------------------
# RAG Function
# -----------------------------
def ask_business_analyst(question, n_results=5):
    # Retrieve relevant reviews
    reviews = retrieve_reviews(question, n_results)

    # Build context
    context = "\\n\\n".join(
        [f"Review {i+1}:\\n{doc}" for i, doc in enumerate(reviews)]
    )

    # Prompt
    prompt = f"""
You are a senior business analyst.

Analyze the following customer reviews and answer the question.

Question:
{question}

Customer Reviews:
{context}

Provide your response in the following format:

1. Executive Summary
2. Key Complaint Themes
3. Business Recommendations
4. Priority Actions
"""

    # Generate response using Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# -----------------------------
# Sidebar
# -----------------------------
st.sidebar.header("📌 Example Questions")

example_questions = [
    "What are the top customer complaints?",
    "Why are customers unhappy?",
    "What should the company improve first?",
    "Summarize the main product defects.",
    "What installation issues are most common?"
]

for q in example_questions:
    st.sidebar.write("• " + q)

# -----------------------------
# Main Input Area
# -----------------------------
default_question = (
    "What are the top customer complaints and "
    "what should the company improve first?"
)

question = st.text_area(
    "Ask a business question:",
    value=default_question,
    height=120
)

# -----------------------------
# Analyze Button
# -----------------------------
if st.button("🚀 Analyze Reviews", use_container_width=True):
    with st.spinner("Analyzing customer reviews..."):
        answer = ask_business_analyst(question)

    st.success("Analysis completed successfully! 🎉")

    st.markdown("## 📊 AI Business Analysis")
    st.markdown(answer)

# -----------------------------
# Footer
# -----------------------------
st.markdown("---")
st.caption(
    "Built by Durgesh Giri using NLP, Sentence Transformers, "
    "Semantic Search, RAG, and Groq LLM."
)
'''

# Write the file
with open(file_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("Streamlit app created successfully! 🚀")
print("Location:", file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Streamlit app created successfully! 🚀
Location: /content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/app/streamlit_app.py


Lauch Streamlit

In [10]:
# Install required packages
!pip install -q streamlit pyngrok

# Create Streamlit secrets file for Groq API key
import os
from google.colab import userdata

# Get Groq API key from Colab Secrets
groq_api_key = userdata.get("GROQ_API_KEY")

# Create .streamlit folder
os.makedirs("/root/.streamlit", exist_ok=True)

# Write secrets.toml
with open("/root/.streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{groq_api_key}"')

print("Streamlit secrets configured successfully! 🔐")

# Start Streamlit in background
import subprocess
subprocess.Popen([
    "streamlit",
    "run",
    "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/app/streamlit_app.py",
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0"
])

print("Streamlit server started successfully! 🚀")

# Create public URL using ngrok
from pyngrok import ngrok

# Kill old tunnels if any
ngrok.kill()

# Create new tunnel
public_url = ngrok.connect(8501)

print("🌐 Open this URL in your browser:")
print(public_url)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.1 MB/s eta 0:00:00
Streamlit secrets configured successfully! 🔐
Streamlit server started successfully! 🚀


ERROR:pyngrok.process.ngrok:t=2026-05-20T06:47:56+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-20T06:47:56+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [11]:
# Install Streamlit and LocalTunnel
!pip install -q streamlit
!npm install -g localtunnel

import os
import subprocess
import time
from google.colab import userdata

# -----------------------------------
# Create Streamlit secrets file
# -----------------------------------
groq_api_key = userdata.get("GROQ_API_KEY")

os.makedirs("/root/.streamlit", exist_ok=True)

with open("/root/.streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{groq_api_key}"')

print("Streamlit secrets configured successfully! 🔐")

# -----------------------------------
# Start Streamlit server
# -----------------------------------
streamlit_process = subprocess.Popen([
    "streamlit",
    "run",
    "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/app/streamlit_app.py",
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0"
])

print("Starting Streamlit server... 🚀")
time.sleep(10)

# -----------------------------------
# Create LocalTunnel URL
# -----------------------------------
tunnel_process = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait until URL is printed
public_url = None
for _ in range(30):
    line = tunnel_process.stdout.readline().strip()
    if "https://" in line:
        public_url = line
        break

print("\n🌐 Open this URL in your browser:")
print(public_url)

print("\n🔑 If LocalTunnel asks for a password, run this command:")
print("!curl https://loca.lt/mytunnelpassword")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 11.14.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.1
npm notice To update run: npm install -g npm@11.14.1
npm notice
⠙Streamlit secrets configured successfully! 🔐
Starting Streamlit server... 🚀

🌐 Open this URL in your browser:
your url is: https://fuzzy-lamps-enjoy.loca.lt

🔑 If LocalTunnel asks for a password, run this command:
!curl https://loca.lt/mytunnelpassword


Automatic Password

In [12]:
!curl https://loca.lt/mytunnelpassword

104.196.107.217

In [13]:
# ==========================================================
# Create api/main.py with complete FastAPI backend
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# Project paths
PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"
API_PATH = os.path.join(PROJECT_PATH, "api")

# Create api folder
os.makedirs(API_PATH, exist_ok=True)

# Full file path
file_path = os.path.join(API_PATH, "main.py")

# Complete FastAPI backend code
fastapi_code = '''
import os
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# --------------------------------------------------
# FastAPI App
# --------------------------------------------------
app = FastAPI(
    title="Customer Review Insight Engine API",
    version="1.0.0"
)

# --------------------------------------------------
# Request Model
# --------------------------------------------------
class QueryRequest(BaseModel):
    question: str
    n_results: int = 5

# --------------------------------------------------
# Load API Key
# --------------------------------------------------
groq_api_key = os.getenv("GROQ_API_KEY")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# --------------------------------------------------
# Load Data and Model on Startup
# --------------------------------------------------
print("Loading resources...")

project_path = os.path.dirname(os.path.dirname(__file__))
data_path = os.path.join(project_path, "data", "negative_reviews.csv")

df = pd.read_csv(data_path)

documents = (
    df["reviewText"]
    .dropna()
    .astype(str)
    .head(5000)
    .tolist()
)

model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = model.encode(
    documents,
    show_progress_bar=False
)

print("Resources loaded successfully!")

# --------------------------------------------------
# Retrieval Function
# --------------------------------------------------
def retrieve_reviews(query, n_results=5):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    top_indices = similarities.argsort()[-n_results:][::-1]

    return [documents[i] for i in top_indices]

# --------------------------------------------------
# RAG Function
# --------------------------------------------------
def generate_analysis(question, n_results=5):
    reviews = retrieve_reviews(question, n_results)

    context = "\\n\\n".join(
        [f"Review {i+1}:\\n{doc}" for i, doc in enumerate(reviews)]
    )

    prompt = f"""
You are a senior business analyst.

Analyze the following customer reviews and answer the question.

Question:
{question}

Customer Reviews:
{context}

Provide:
1. Executive Summary
2. Key Complaint Themes
3. Business Recommendations
4. Priority Actions
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# --------------------------------------------------
# Health Check Endpoint
# --------------------------------------------------
@app.get("/")
def home():
    return {
        "message": "Customer Review Insight Engine API is running!"
    }

# --------------------------------------------------
# Analysis Endpoint
# --------------------------------------------------
@app.post("/analyze")
def analyze(request: QueryRequest):
    answer = generate_analysis(
        request.question,
        request.n_results
    )

    return {
        "question": request.question,
        "analysis": answer
    }
'''

# Write file
with open(file_path, "w", encoding="utf-8") as f:
    f.write(fastapi_code)

print("FastAPI backend created successfully! 🚀")
print("Location:", file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FastAPI backend created successfully! 🚀
Location: /content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/api/main.py


In [1]:
!pip install -q fastapi uvicorn

Run FastAPI Server in Colab

In [2]:
# Install LocalTunnel (if not already installed)
!npm install -g localtunnel

import os
import subprocess
import time
from google.colab import userdata

# --------------------------------------------------
# Set GROQ_API_KEY as environment variable
# --------------------------------------------------
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("GROQ_API_KEY loaded successfully! 🔐")

# --------------------------------------------------
# Path to API folder
# --------------------------------------------------
API_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/api"

# --------------------------------------------------
# Start FastAPI using Uvicorn
# --------------------------------------------------
uvicorn_process = subprocess.Popen([
    "uvicorn",
    "main:app",
    "--host", "0.0.0.0",
    "--port", "8000"
], cwd=API_PATH)

print("Starting FastAPI server... 🚀")
time.sleep(10)

# --------------------------------------------------
# Create LocalTunnel URL
# --------------------------------------------------
tunnel_process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(30):
    line = tunnel_process.stdout.readline().strip()
    if "https://" in line:
        public_url = line
        break

print("\n🌐 FastAPI Public URL:")
print(public_url)

print("\n📘 Swagger UI:")
print(public_url + "/docs")

print("\n🔑 If LocalTunnel asks for password, run:")
print("!curl https://loca.lt/mytunnelpassword")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙GROQ_API_KEY loaded successfully! 🔐


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/api'

Check whether api/main.py exists

In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"
API_PATH = os.path.join(PROJECT_PATH, "api")
MAIN_FILE = os.path.join(API_PATH, "main.py")

print("Project exists:", os.path.exists(PROJECT_PATH))
print("API folder exists:", os.path.exists(API_PATH))
print("main.py exists:", os.path.exists(MAIN_FILE))

if os.path.exists(PROJECT_PATH):
    print("\nProject contents:")
    print(os.listdir(PROJECT_PATH))

Project exists: False
API folder exists: False
main.py exists: False


Find the Exact Path

In [4]:
import os

# List MyDrive contents
print("Folders in MyDrive:\n")
for item in os.listdir("/content/drive/MyDrive"):
    print(item)

Folders in MyDrive:



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

Mount Google Drive Again

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Verify Project Path Automatically

In [6]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "Customer_Review_Insight_Engine" in dirs:
        print("Project found at:")
        print(os.path.join(root, "Customer_Review_Insight_Engine"))
        break

Project found at:
/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine


Step 6.2: Run FastAPI Server

In [7]:
# Install LocalTunnel if needed
!npm install -g localtunnel

import os
import subprocess
import time
from google.colab import userdata

# --------------------------------------------------
# Load GROQ API Key
# --------------------------------------------------
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("GROQ_API_KEY loaded successfully! 🔐")

# --------------------------------------------------
# Define API Path
# --------------------------------------------------
API_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/api"
MAIN_FILE = os.path.join(API_PATH, "main.py")

# Verify file exists
print("API folder exists:", os.path.exists(API_PATH))
print("main.py exists:", os.path.exists(MAIN_FILE))

# --------------------------------------------------
# Start FastAPI Server
# --------------------------------------------------
uvicorn_process = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd=API_PATH
)

print("Starting FastAPI server... 🚀")
time.sleep(15)

# --------------------------------------------------
# Create LocalTunnel URL
# --------------------------------------------------
tunnel_process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(30):
    line = tunnel_process.stdout.readline().strip()
    if "https://" in line:
        # LocalTunnel prints: "your url is: https://...."
        public_url = line.split()[-1]
        break

print("\n🌐 FastAPI Public URL:")
print(public_url)

print("\n📘 Swagger UI:")
print(public_url + "/docs")

print("\n🔑 If LocalTunnel asks for a password, run:")
print("!curl https://loca.lt/mytunnelpassword")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏GROQ_API_KEY loaded successfully! 🔐
API folder exists: True
main.py exists: True
Starting FastAPI server... 🚀

🌐 FastAPI Public URL:
https://two-crabs-roll.loca.lt

📘 Swagger UI:
https://two-crabs-roll.loca.lt/docs

🔑 If LocalTunnel asks for a password, run:
!curl https://loca.lt/mytunnelpassword


In [8]:
!curl https://loca.lt/mytunnelpassword

34.24.16.135

In [9]:
# Install dependencies
!pip install -q fastapi uvicorn
!npm install -g localtunnel

import os
import subprocess
import time
from google.colab import userdata

# ----------------------------------------
# Set Groq API key
# ----------------------------------------
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("GROQ_API_KEY loaded successfully! 🔐")

# ----------------------------------------
# Define API path
# ----------------------------------------
API_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine/api"
MAIN_FILE = os.path.join(API_PATH, "main.py")

print("API folder exists:", os.path.exists(API_PATH))
print("main.py exists:", os.path.exists(MAIN_FILE))

# ----------------------------------------
# Kill old processes (optional cleanup)
# ----------------------------------------
!pkill -f uvicorn || true
!pkill -f lt || true

# ----------------------------------------
# Start FastAPI server
# ----------------------------------------
uvicorn_process = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd=API_PATH
)

print("Starting FastAPI server... 🚀")
time.sleep(20)   # give enough time for model loading

# ----------------------------------------
# Start LocalTunnel
# ----------------------------------------
tunnel_process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(60):
    line = tunnel_process.stdout.readline().strip()
    if "https://" in line:
        public_url = line.split()[-1]
        break

print("\n🌐 FastAPI Public URL:")
print(public_url)

if public_url:
    print("\n📘 Swagger UI:")
    print(public_url + "/docs")

print("\n🔑 If LocalTunnel asks for a password, run:")
print("!curl https://loca.lt/mytunnelpassword")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧GROQ_API_KEY loaded successfully! 🔐
API folder exists: True
main.py exists: True
^C
^C
Starting FastAPI server... 🚀

🌐 FastAPI Public URL:
https://metal-falcons-matter.loca.lt

📘 Swagger UI:
https://metal-falcons-matter.loca.lt/docs

🔑 If LocalTunnel asks for a password, run:
!curl https://loca.lt/mytunnelpassword


In [10]:
!curl https://loca.lt/mytunnelpassword

34.24.16.135

In [ ]:
Step 7. Create Docker Files

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Project path
PROJECT_PATH = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"

# Ensure project folder exists
os.makedirs(PROJECT_PATH, exist_ok=True)

# ---------------------------------------------------
# requirements.txt
# ---------------------------------------------------
requirements_txt = """
pandas
numpy
streamlit
fastapi
uvicorn
sentence-transformers
scikit-learn
openai
chromadb
"""

# ---------------------------------------------------
# Dockerfile
# ---------------------------------------------------
dockerfile = """
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501
EXPOSE 8000

CMD ["streamlit", "run", "app/streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
"""

# ---------------------------------------------------
# .dockerignore
# ---------------------------------------------------
dockerignore = """
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.venv/
"""

# ---------------------------------------------------
# docker-compose.yml
# ---------------------------------------------------
docker_compose = """
version: '3.9'

services:
  customer-review-insight-engine:
    build: .
    ports:
      - "8501:8501"
      - "8000:8000"
    environment:
      - GROQ_API_KEY=${GROQ_API_KEY}
"""

# ---------------------------------------------------
# Write files
# ---------------------------------------------------
files = {
    "requirements.txt": requirements_txt.strip(),
    "Dockerfile": dockerfile.strip(),
    ".dockerignore": dockerignore.strip(),
    "docker-compose.yml": docker_compose.strip(),
}

for filename, content in files.items():
    path = os.path.join(PROJECT_PATH, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

print("Docker files created successfully! 🐳🚀")
print("Location:", PROJECT_PATH)

print("\nCreated files:")
for filename in files:
    print(" -", filename)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Docker files created successfully! 🐳🚀
Location: /content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine

Created files:
 - requirements.txt
 - Dockerfile
 - .dockerignore
 - docker-compose.yml


🧠 What Each File Does
File	Purpose
Dockerfile	Defines how to build the container
requirements.txt	Lists Python dependencies
.dockerignore	Excludes unnecessary files
docker-compose.yml	Makes running the project easier

How to Run later on your Laptop

In [13]:
!docker build -t customer-review-insight-engine .
!docker run -p 8501:8501 -e GROQ_API_KEY=your_key customer-review-insight-engine

/bin/bash: line 1: docker: command not found
/bin/bash: line 1: docker: command not found


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Source directory (where your scattered files currently are)
SOURCE_DIR = "/content/drive/MyDrive"

# Project root
PROJECT_ROOT = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"

# Create folders
folders = [
    "data",
    "notebooks",
    "app",
    "api"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print("Project folders created successfully! 📁")

# File mapping: source filename -> destination relative path
file_mapping = {
    # Data files
    "amazon_reviews.csv": "data/amazon_reviews.csv",
    "negative_reviews.csv": "data/negative_reviews.csv",

    # Notebooks
    "02_eda_and_business_understanding.ipynb": "notebooks/02_eda_and_business_understanding.ipynb",
    "03_semantic_search_with_chromadb.ipynb": "notebooks/03_semantic_search_with_chromadb.ipynb",
    "04_rag_with_openai.ipynb": "notebooks/04_rag_with_openai.ipynb",
    "AI-Powered_Product_Review_Intelligence_Platform.ipynb":
        "notebooks/AI-Powered_Product_Review_Intelligence_Platform.ipynb",

    # Application files
    "streamlit_app.py": "app/streamlit_app.py",
    "main.py": "api/main.py",

    # Docker files
    "Dockerfile": "Dockerfile",
    "requirements.txt": "requirements.txt",
    "docker-compose.yml": "docker-compose.yml",
    ".dockerignore": ".dockerignore",
}

# Move files if found
for filename, relative_dest in file_mapping.items():
    src = os.path.join(SOURCE_DIR, filename)
    dst = os.path.join(PROJECT_ROOT, relative_dest)

    if os.path.exists(src):
        shutil.move(src, dst)
        print(f"✅ Moved: {filename} → {relative_dest}")
    else:
        print(f"⚠️ Not found: {filename}")

print("\n🎉 Project organized successfully!")
print("📂 Final location:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders created successfully! 📁
⚠️ Not found: amazon_reviews.csv
⚠️ Not found: negative_reviews.csv
⚠️ Not found: 02_eda_and_business_understanding.ipynb
⚠️ Not found: 03_semantic_search_with_chromadb.ipynb
⚠️ Not found: 04_rag_with_openai.ipynb
⚠️ Not found: AI-Powered_Product_Review_Intelligence_Platform.ipynb
⚠️ Not found: streamlit_app.py
⚠️ Not found: main.py
⚠️ Not found: Dockerfile
⚠️ Not found: requirements.txt
⚠️ Not found: docker-compose.yml
⚠️ Not found: .dockerignore

🎉 Project organized successfully!
📂 Final location:
/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine


In [15]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Project root
PROJECT_ROOT = "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine"

# Required folders
folders = ["data", "notebooks", "app", "api"]
for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print("Project folders ensured successfully! 📁")

# File mapping: filename -> destination path inside project
file_mapping = {
    # Data
    "amazon_reviews.csv": "data/amazon_reviews.csv",
    "negative_reviews.csv": "data/negative_reviews.csv",

    # Notebooks
    "02_eda_and_business_understanding.ipynb": "notebooks/02_eda_and_business_understanding.ipynb",
    "02_eda_and_business_understanding.ipynb.ipynb": "notebooks/02_eda_and_business_understanding.ipynb",
    "03_semantic_search_with_chromadb.ipynb": "notebooks/03_semantic_search_with_chromadb.ipynb",
    "04_rag_with_openai.ipynb": "notebooks/04_rag_with_openai.ipynb",
    "AI-Powered_Product_Review_Intelligence_Platform.ipynb":
        "notebooks/AI-Powered_Product_Review_Intelligence_Platform.ipynb",

    # App / API
    "streamlit_app.py": "app/streamlit_app.py",
    "main.py": "api/main.py",

    # Docker
    "Dockerfile": "Dockerfile",
    "requirements.txt": "requirements.txt",
    "docker-compose.yml": "docker-compose.yml",
    ".dockerignore": ".dockerignore",
}

# Search recursively in MyDrive
search_root = "/content/drive/MyDrive"

def find_file(filename):
    for root, dirs, files in os.walk(search_root):
        if filename in files:
            return os.path.join(root, filename)
    return None

# Move files
for filename, relative_dest in file_mapping.items():
    src = find_file(filename)
    dst = os.path.join(PROJECT_ROOT, relative_dest)

    if src:
        # Skip if source and destination are same
        if os.path.abspath(src) != os.path.abspath(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)   # copy instead of move for safety
            print(f"✅ Copied: {filename} → {relative_dest}")
        else:
            print(f"✔ Already in correct location: {filename}")
    else:
        print(f"⚠️ Not found anywhere in MyDrive: {filename}")

print("\n🎉 Project organized successfully!")
print("📂 Final location:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders ensured successfully! 📁
✔ Already in correct location: amazon_reviews.csv
✔ Already in correct location: negative_reviews.csv
⚠️ Not found anywhere in MyDrive: 02_eda_and_business_understanding.ipynb
✅ Copied: 02_eda_and_business_understanding.ipynb.ipynb → notebooks/02_eda_and_business_understanding.ipynb
✔ Already in correct location: 03_semantic_search_with_chromadb.ipynb
✔ Already in correct location: 04_rag_with_openai.ipynb
⚠️ Not found anywhere in MyDrive: AI-Powered_Product_Review_Intelligence_Platform.ipynb
✔ Already in correct location: streamlit_app.py
✔ Already in correct location: main.py
✔ Already in correct location: Dockerfile
✔ Already in correct location: requirements.txt
✔ Already in correct location: docker-compose.yml
✔ Already in correct location: .dockerignore

🎉 Project organized successfully!
📂 Final location:
/content

In [17]:
!apt-get -qq install tree
!tree "/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine" -L 3

Selecting previously unselected package tree.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
/content/drive/MyDrive/AI_Projects/Customer_Review_Insight_Engine
├── api
│   ├── main.py
│   └── __pycache__
│       └── main.cpython-312.pyc
├── app
│   └── streamlit_app.py
├── data
│   ├── amazon_reviews.csv
│   ├── amazon_reviews_processed.csv
│   ├── iris-data.csv
│   ├── negative_review_clusters.csv
│   ├── negative_reviews.csv
│   └── top_complaint_keywords.csv
├── docker-compose.yml
├── Dockerfile
├── docs
├── models
├── notebooks
│   ├── 02_eda_and_business_understanding.ipynb
│   ├── 03_semantic_search_with_chromadb.ipynb
│   └── 04_rag_with_openai.ipynb
├── outputs
├── requirements.txt
└── src

9 directories, 15 files
